# Array operations and statistics

## Learning objectives

By the end of this notebook you will be able to:

- do element-wise arithmetic and use universal functions
- explain and use broadcasting
- compute statistics for a whole array and along an axis
- standardise data (z-scores) and compute percentiles
- handle NaN and infinity correctly

## Introduction

Most numerical work is a few operations applied to many numbers: convert units, subtract a baseline,
average replicates, standardise. NumPy does each of these in one line, over whole arrays.

In [1]:
import numpy as np

## Element-wise arithmetic

Operations between arrays of the same shape pair up elements by position.

In [2]:
before = np.array([142, 128, 155, 118, 149])
after = np.array([131, 126, 141, 117, 138])
change = after - before
print(change)
print((change / before * 100).round(1))

[-11  -2 -14  -1 -11]
[-7.7 -1.6 -9.  -0.8 -7.4]


## Universal functions (ufuncs)

Functions that act on every element.

In [3]:
x = np.array([1, 10, 100, 1000])
print(np.sqrt(x), np.log10(x), np.log(x).round(2), np.exp([0, 1]).round(3))
print(np.abs([-2, 3]), np.round([2.456, 3.14159], 2), np.floor([2.7]), np.ceil([2.1]))

[ 1.          3.16227766 10.         31.6227766 ] [0. 1. 2. 3.] [0.   2.3  4.61 6.91] [1.    2.718]
[2 3] [2.46 3.14] [2.] [3.]


Log transforms are common for skewed lab values such as concentrations or viral loads:

In [4]:
viral_load = np.array([150, 2_400, 38_000, 910_000, 12_000_000])
print(np.log10(viral_load).round(2))

[2.18 3.38 4.58 5.96 7.08]


## Broadcasting

When shapes differ, NumPy stretches the smaller array to match, if the shapes are compatible. Shapes are
compared from the right; each pair of sizes must be equal, or one of them must be 1.

| shapes | result |
|---|---|
| (5,) and scalar | (5,) |
| (6, 4) and (4,) | (6, 4): the row is repeated for every row |
| (6, 4) and (6, 1) | (6, 4): the column is repeated for every column |
| (6, 4) and (6,) | error: compares 4 with 6 |

In [5]:
sbp = np.array([
    [142, 138, 135, 131],
    [128, 130, 127, 126],
    [155, 150, 146, 141],
    [118, 121, 119, 117],
    [149, 147, 140, 138],
    [133, 129, 128, 124],
])
baseline = sbp[:, 0:1]            # shape (6, 1)
print(sbp - baseline)             # change from each participant's own baseline

[[  0  -4  -7 -11]
 [  0   2  -1  -2]
 [  0  -5  -9 -14]
 [  0   3   1  -1]
 [  0  -2  -9 -11]
 [  0  -4  -5  -9]]


In [6]:
visit_means = sbp.mean(axis=0)   # shape (4,)
print(visit_means.round(1))
print((sbp - visit_means).round(1))   # deviation from each visit's mean

[137.5 135.8 132.5 129.5]
[[  4.5   2.2   2.5   1.5]
 [ -9.5  -5.8  -5.5  -3.5]
 [ 17.5  14.2  13.5  11.5]
 [-19.5 -14.8 -13.5 -12.5]
 [ 11.5  11.2   7.5   8.5]
 [ -4.5  -6.8  -4.5  -5.5]]


In [7]:
try:
    sbp - np.arange(6)
except ValueError as e:
    print("ValueError:", e)

ValueError: operands could not be broadcast together with shapes (6,4) (6,) 


## Statistical functions

| function | |
|---|---|
| `sum`, `mean`, `median` | totals and centre |
| `min`, `max`, `ptp` | extremes and range |
| `std`, `var` | spread (`ddof=1` for the sample version) |
| `percentile`, `quantile` | cut points |
| `argmin`, `argmax` | position of the extreme |
| `cumsum` | running total |
| `corrcoef` | correlation |

In [8]:
rng = np.random.default_rng(5)
hb = rng.normal(13.5, 1.6, 300)
print(f"mean {hb.mean():.2f}  median {np.median(hb):.2f}")
print(f"sd {hb.std(ddof=1):.2f}  variance {hb.var(ddof=1):.2f}")
print(f"min {hb.min():.1f}  max {hb.max():.1f}  range {np.ptp(hb):.1f}")
print("quartiles", np.percentile(hb, [25, 50, 75]).round(2))

mean 13.44  median 13.39
sd 1.54  variance 2.36
min 9.7  max 17.9  range 8.3
quartiles [12.37 13.39 14.51]


`ddof=1` divides by n − 1, giving the **sample** standard deviation, which is what statistics software
reports. NumPy's default (`ddof=0`) divides by n. With large n they barely differ; with small n they do.

In [9]:
small = np.array([4.0, 6.0, 8.0])
print(small.std(), small.std(ddof=1))

1.632993161855452 2.0


## Statistics along an axis

`axis=0` works **down the rows** (one result per column); `axis=1` works **across the columns** (one
result per row).

```
        axis=1 →
axis=0  [[142 138 135 131]
  ↓      [128 130 127 126] ...
```

In [10]:
print("mean per visit      :", sbp.mean(axis=0).round(1))
print("mean per participant:", sbp.mean(axis=1).round(1))
print("highest per visit   :", sbp.max(axis=0))
print("visit of each participant's peak:", sbp.argmax(axis=1))

mean per visit      : [137.5 135.8 132.5 129.5]
mean per participant: [136.5 127.8 148.  118.8 143.5 128.5]
highest per visit   : [155 150 146 141]
visit of each participant's peak: [0 1 0 1 0 0]


Reading this: blood pressure falls on average from the first to the last visit, and every participant's
highest reading is at the first visit except participants 2 and 4, whose second visits were highest.

## Standardisation: z-scores

A z-score says how many standard deviations a value lies from the mean. It puts measurements on
different scales onto a common one.

In [11]:
z = (hb - hb.mean()) / hb.std(ddof=1)
print(z[:5].round(2))
print(f"mean of z {z.mean():.3f}, sd of z {z.std(ddof=1):.3f}")
print(f"values more than 2 sd from the mean: {(np.abs(z) > 2).sum()} of {len(z)}")

[-0.79 -1.34 -0.22  0.48  1.22]
mean of z -0.000, sd of z 1.000
values more than 2 sd from the mean: 14 of 300


For normally distributed data, about 5% of values lie more than 2 standard deviations from the mean, so
roughly 15 of 300 is expected.

## Correlation

In [12]:
weight = rng.normal(70, 12, 100)
height = 1.2 + weight / 100 + rng.normal(0, 0.06, 100)
r = np.corrcoef(weight, height)
print(r.round(3))
print(f"r = {r[0, 1]:.2f}")

[[1.    0.899]
 [0.899 1.   ]]
r = 0.90


`corrcoef` returns a matrix: the diagonal is each variable with itself (1), the off-diagonal value is
the correlation between the two.

## Missing values: NaN

`np.nan` (Not a Number) marks a missing value. Any calculation involving NaN gives NaN, so ordinary
functions return NaN when a single value is missing. The `nan`-functions ignore missing values.

In [13]:
results = np.array([5.4, np.nan, 6.1, 7.8, np.nan, 6.6])
print(results.mean(), np.nanmean(results).round(2), np.nanmedian(results), np.nanstd(results, ddof=1).round(3))
print("missing:", np.isnan(results).sum())
print("complete values:", results[~np.isnan(results)])

nan 6.48 6.35 1.011
missing: 2
complete values: [5.4 6.1 7.8 6.6]


NaN is not equal to anything, not even itself, so `results == np.nan` never finds it. Always use `np.isnan`.

In [14]:
print(np.nan == np.nan, (results == np.nan).sum(), np.isnan(results).sum())

False 0 2


Replacing missing values (only after deciding that this is appropriate for the analysis):

In [15]:
filled = np.where(np.isnan(results), np.nanmedian(results), results)
print(filled)

[5.4  6.35 6.1  7.8  6.35 6.6 ]


## Infinity

Dividing a non-zero number by zero gives `inf`; zero by zero gives `nan`. NumPy warns instead of
stopping. Log of zero is `-inf`.

In [16]:
with np.errstate(divide="ignore", invalid="ignore"):
    ratio = np.array([4.0, 3.0, 0.0]) / np.array([2.0, 0.0, 0.0])
    logs = np.log10(np.array([100.0, 0.0]))
print(ratio, logs)
print(np.isinf(ratio), np.isfinite(ratio))
print(ratio[np.isfinite(ratio)])

[ 2. inf nan] [  2. -inf]
[False  True False] [ True False False]
[2.]


`np.errstate` silences the warnings for that block only, because here we expect them. In real work,
the warnings are useful: they point at zeros you may not have known about.

## Practise

Exercises for this folder are in `exercises/NumPy_Exercises.ipynb`.

## Further reading

- [NumPy user guide](https://numpy.org/doc/stable/user/index.html)
- [Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html)
- [Statistics routines](https://numpy.org/doc/stable/reference/routines.statistics.html)
- [Universal functions](https://numpy.org/doc/stable/reference/ufuncs.html)